# Administração de Redes: Trabalho 2
## Montagem e análise do laboratório OSPF no GNS3 (2 AS, 3 áreas)

**Mestrado em Engenharia de Redes, FCUP, 2026/27**

Este documento regista **como o laboratório foi montado e validado**: plano de endereçamento, configuração de cada router, comandos de verificação e os erros encontrados pelo caminho (com a causa e a correção). As respostas às questões 1 a 7 do enunciado estão na **secção 7**, com as capturas feitas sobre esta rede.

> **Estado atual (2026-10-18):** rede convergida. Ping e traceroute entre Terminal 1 e Terminal 2 funcionam nos dois sentidos, e todos os routers têm a configuração gravada (`write memory`). Questões 1 a 7 respondidas (secção 7).


## 0. Topologia, áreas e plano de endereçamento

Duas redes autónomas ligadas pelo R2:

- **AS1** (área 0): R1 e R2.
- **AS2**: R2 (lado AS2), R3, R4, R5 e R6 na área 0; R6, R7 e R9 na área 1; R6, R8 e R9 na área 2.
- **R6** é o ABR das áreas 0, 1 e 2. **R9** liga as áreas 1 e 2 e não tem ligação direta à área 0. **R2** é ASBR e corre dois processos OSPF.

### Ligações e endereços (portas reais do projeto GNS3)

| Router | Porta | Liga a | Endereço | Rede | Área |
|---|---|---|---|---|---|
| R1 | Fa1/1 | R2 | 192.168.1.129/26 | 192.168.1.128/26 | AS1, 0 |
| R1 | Fa1/0 | Terminal 1 | 192.168.1.1/25 | 192.168.1.0/25 | AS1, 0 |
| R2 | Fa1/1 | R1 | 192.168.1.130/26 | 192.168.1.128/26 | AS1, 0 |
| R2 | Fa1/0 | C1 | 192.168.1.193/26 | 192.168.1.192/26 | AS1, 0 |
| R2 | Fa2/0 | R3 | 192.168.50.1/24 | 192.168.50.0/24 | AS2, 0 |
| R3 | Fa1/1 | R2 | 192.168.50.2/24 | 192.168.50.0/24 | AS2, 0 |
| R3 | Fa1/0 | Nuvem | 192.168.60.1/24 | 192.168.60.0/24 | AS2, 0 |
| R3 | Fa2/0 | Switch | 192.168.100.1/24 | 192.168.100.0/24 | AS2, 0 |
| R4 | Fa1/0 | Switch | 192.168.100.253/24 | 192.168.100.0/24 | AS2, 0 |
| R5 | Fa1/0 | Switch | 192.168.100.254/24 | 192.168.100.0/24 | AS2, 0 |
| R6 | Fa1/0 | Switch | 192.168.100.2/24 | 192.168.100.0/24 | AS2, 0 |
| R6 | Fa1/1 | R7 | 172.20.1.1/30 | 172.20.1.0/30 | AS2, 1 |
| R6 | Se2/0 | R8 | 172.20.1.14/30 | 172.20.1.12/30 | AS2, 2 |
| R7 | Fa1/0 | R6 | 172.20.1.2/30 | 172.20.1.0/30 | AS2, 1 |
| R7 | Fa1/1 | R9 | 172.20.1.5/30 | 172.20.1.4/30 | AS2, 1 |
| R7 | Fa2/0 | Terminal 2 | 172.16.1.1/24 | 172.16.1.0/24 | AS2, 1 |
| R8 | Se2/0 | R6 | 172.20.1.13/30 | 172.20.1.12/30 | AS2, 2 |
| R8 | Fa1/1 | R9 | 172.20.1.10/30 | 172.20.1.8/30 | AS2, 2 |
| R8 | Fa1/0 | C3 | 172.18.1.1/24 | 172.18.1.0/24 | AS2, 2 |
| R9 | Fa1/0 | R7 | 172.20.1.6/30 | 172.20.1.4/30 | AS2, 1 |
| R9 | Fa1/1 | R8 | 172.20.1.9/30 | 172.20.1.8/30 | AS2, 2 |
| R9 | Fa2/0 | C4 | 172.17.0.1/16 | 172.17.0.0/16 | AS2, 2 |

**Terminais (VPCS)**

| Terminal | Endereço | Gateway |
|---|---|---|
| Terminal 1 | 192.168.1.123/25 | 192.168.1.1 |
| Terminal 2 | 172.16.1.2/24 | 172.16.1.1 |

**Router IDs** (regra do enunciado): `1.0.0.n` no AS1 e `2.0.0.n` no AS2. O R2 tem dois: `1.0.0.2` (processo 1, AS1) e `2.0.0.2` (processo 2, AS2).

![Planta da rede do enunciado (AS1, AS2 e áreas OSPF)](evidencias/t2_00_topologia.png)

*Planta da rede do enunciado (AS1, AS2 e áreas OSPF)*


## 1. Como se calculam máscaras e wildcards

Os prefixos `/25`, `/26`, `/24` e `/16` vêm escritos nas nuvens da figura. Os das ligações `172.20.1.x` e `192.168.100.x` não vêm, e foram **inferidos**:

- **172.20.1.x: /30.** Cada ligação tem só dois routers, e os pares de endereços ficam em blocos consecutivos de 4 (.0-.3, .4-.7, .8-.11, .12-.15).
- **192.168.100.0: /24.** Os endereços .1, .2, .253 e .254 encaixam naturalmente.

Se a máscara diferir entre dois vizinhos, a adjacência fica presa em `INIT` ou `EXSTART`.

**Método (exemplo /26)**

1. Bits de host = 32 menos o prefixo = 6.
2. Tamanho do bloco = 2^6 = 64.
3. Máscara no último octeto = 256 menos o bloco = 192, logo `255.255.255.192`.
4. **Wildcard** (inverso da máscara) = 255 menos cada octeto = bloco menos 1 = `0.0.0.63`.

| Prefixo | Máscara | Wildcard | Bloco |
|---|---|---|---|
| /16 | 255.255.0.0 | 0.0.255.255 | 65536 |
| /24 | 255.255.255.0 | 0.0.0.255 | 256 |
| /25 | 255.255.255.128 | 0.0.0.127 | 128 |
| /26 | 255.255.255.192 | 0.0.0.63 | 64 |
| /30 | 255.255.255.252 | 0.0.0.3 | 4 |

**Para achar o endereço de rede** (o que o comando `network` pede), localize o endereço no bloco: com /30, 172.20.1.5 e 172.20.1.6 caem em .4-.7, logo a rede é `172.20.1.4`. Na wildcard, `0` significa "este bit tem de coincidir" e `1` significa "ignorar".

Os três blocos do AS1 não se sobrepõem e preenchem 192.168.1.0/24: `.0/25` (.0-.127), `.128/26` (.128-.191) e `.192/26` (.192-.255). É por isso que o enunciado manda sumarizá-los em `192.168.1.0/24`.

## 2. Regras transversais da configuração

- **`auto-cost reference-bandwidth 1000`** em todos os routers (e nos dois processos do R2). Com isto cada FastEthernet custa 10 (1000/100 Mbps), como o enunciado pede.
- **`passive-interface default`** seguido de `no passive-interface <if>` só nas interfaces que ligam a outros routers. As LANs com terminais ou nuvens continuam anunciadas pelo `network`, mas não enviam hellos OSPF.
- **`router-id`** fixado à mão em cada processo.
- **Número do processo OSPF:** é local ao router. Usou-se 1 para o AS1 e 2 para o AS2, também no R2.
- **`no logging console`** durante a configuração, para as mensagens `%OSPF-4-ERRRCV` não interromperem a escrita (ver erro 2 na secção 5).
- **`write memory`** em todos os routers no fim. Sem gravar, a configuração perde-se ao parar o router no GNS3.

Convenção dos blocos: a linha `network` define em que área fica cada interface e a linha `no passive-interface` define se envia hellos. **São independentes, e as duas têm de existir.**

## 3. Configuração por router (confRes)

Cada bloco parte do modo EXEC privilegiado. Se o prompt for `Router>`, escreve-se primeiro `enable` e `configure terminal`.

### 3.1 R1 (AS1)

```
enable
configure terminal
hostname R1
no ip domain-lookup
interface FastEthernet1/1
 description Link to R2
 ip address 192.168.1.129 255.255.255.192
 no shutdown
 exit
interface FastEthernet1/0
 description LAN Terminal 1
 ip address 192.168.1.1 255.255.255.128
 no shutdown
 exit
router ospf 1
 router-id 1.0.0.1
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 network 192.168.1.128 0.0.0.63 area 0
 network 192.168.1.0 0.0.0.127 area 0
 end
write memory
```

**Terminal 1 (VPCS):** `ip 192.168.1.123/25 192.168.1.1` e `save`.

### 3.2 R2 (ASBR, dois processos, redistribuição mútua e sumarização)

```
enable
configure terminal
hostname R2
no ip domain-lookup
interface FastEthernet1/1
 description Link to R1
 ip address 192.168.1.130 255.255.255.192
 no shutdown
 exit
interface FastEthernet1/0
 description LAN C1
 ip address 192.168.1.193 255.255.255.192
 no shutdown
 exit
interface FastEthernet2/0
 description Link to R3
 ip address 192.168.50.1 255.255.255.0
 no shutdown
 exit
router ospf 1
 router-id 1.0.0.2
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 network 192.168.1.128 0.0.0.63 area 0
 network 192.168.1.192 0.0.0.63 area 0
 redistribute ospf 2 metric 100 subnets
 exit
router ospf 2
 router-id 2.0.0.2
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet2/0
 network 192.168.50.0 0.0.0.255 area 0
 redistribute ospf 1 metric 100 subnets
 summary-address 192.168.1.0 255.255.255.0
 end
write memory
```

- `redistribute ... metric 100` cumpre "rotas importadas com métrica 100". `subnets` é obrigatório, senão só se redistribuem redes classful.
- `summary-address 192.168.1.0 255.255.255.0` sumariza as redes 192.168.1.x **apenas ao serem injetadas no AS2**. Dentro do AS1 mantêm as máscaras originais. O IOS cria automaticamente uma rota de descarte `192.168.1.0/24 ... Null0`.

### 3.3 R3

```
enable
configure terminal
hostname R3
no ip domain-lookup
interface FastEthernet1/1
 description Link to R2
 ip address 192.168.50.2 255.255.255.0
 no shutdown
 exit
interface FastEthernet1/0
 description LAN 192.168.60.0
 ip address 192.168.60.1 255.255.255.0
 no shutdown
 exit
interface FastEthernet2/0
 description Link to switch
 ip address 192.168.100.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.3
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/1
 no passive-interface FastEthernet2/0
 network 192.168.50.0 0.0.0.255 area 0
 network 192.168.60.0 0.0.0.255 area 0
 network 192.168.100.0 0.0.0.255 area 0
 end
write memory
```

### 3.4 R4 e R5 (iguais, mudam endereço, router ID e hostname)

R4 usa `192.168.100.253` e `router-id 2.0.0.4`. R5 usa `192.168.100.254` e `router-id 2.0.0.5`.

```
enable
configure terminal
hostname R4
no ip domain-lookup
interface FastEthernet1/0
 description Link to switch
 ip address 192.168.100.253 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.4
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 network 192.168.100.0 0.0.0.255 area 0
 end
write memory
```

### 3.5 R6 (ABR das áreas 0, 1 e 2)

```
enable
configure terminal
no logging console
hostname R6
no ip domain-lookup
interface FastEthernet1/0
 description Link to switch
 ip address 192.168.100.2 255.255.255.0
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R7
 ip address 172.20.1.1 255.255.255.252
 no shutdown
 exit
interface Serial2/0
 description Link to R8
 ip address 172.20.1.14 255.255.255.252
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.6
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 no passive-interface Serial2/0
 network 192.168.100.0 0.0.0.255 area 0
 network 172.20.1.0 0.0.0.3 area 1
 network 172.20.1.12 0.0.0.3 area 2
 end
write memory
```

A interface série ficou com custo 647 (largura de banda por omissão de 1544 kbps contra a referência de 1000 Mbps). É o esperado, porque o enunciado só fixa o custo das FastEthernet. Não foi preciso `clock rate`.

### 3.6 R7 (área 1)

```
enable
configure terminal
no logging console
hostname R7
no ip domain-lookup
interface FastEthernet1/0
 description Link to R6
 ip address 172.20.1.2 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R9
 ip address 172.20.1.5 255.255.255.252
 no shutdown
 exit
interface FastEthernet2/0
 description LAN Terminal 2
 ip address 172.16.1.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.7
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 network 172.20.1.0 0.0.0.3 area 1
 network 172.20.1.4 0.0.0.3 area 1
 network 172.16.1.0 0.0.0.255 area 1
 end
write memory
```

**Terminal 2 (VPCS):** `ip 172.16.1.2/24 172.16.1.1` e `save`.

### 3.7 R8 (área 2)

```
enable
configure terminal
no logging console
hostname R8
no ip domain-lookup
interface Serial2/0
 description Link to R6
 ip address 172.20.1.13 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R9
 ip address 172.20.1.10 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/0
 description LAN C3
 ip address 172.18.1.1 255.255.255.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.8
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface Serial2/0
 no passive-interface FastEthernet1/1
 network 172.20.1.12 0.0.0.3 area 2
 network 172.20.1.8 0.0.0.3 area 2
 network 172.18.1.0 0.0.0.255 area 2
 end
write memory
```

### 3.8 R9 (áreas 1 e 2, sem ligação direta à área 0)

```
enable
configure terminal
no logging console
hostname R9
no ip domain-lookup
interface FastEthernet1/0
 description Link to R7
 ip address 172.20.1.6 255.255.255.252
 no shutdown
 exit
interface FastEthernet1/1
 description Link to R8
 ip address 172.20.1.9 255.255.255.252
 no shutdown
 exit
interface FastEthernet2/0
 description LAN C4
 ip address 172.17.0.1 255.255.0.0
 no shutdown
 exit
router ospf 2
 router-id 2.0.0.9
 auto-cost reference-bandwidth 1000
 passive-interface default
 no passive-interface FastEthernet1/0
 no passive-interface FastEthernet1/1
 network 172.20.1.4 0.0.0.3 area 1
 network 172.20.1.8 0.0.0.3 area 2
 network 172.17.0.0 0.0.255.255 area 2
 end
write memory
```

## 4. Comandos de verificação (outRes)

Em cada router, depois de configurar:

```
show ip interface brief
show ip ospf interface brief
show ip ospf neighbor
show ip route
show running-config | section ospf
```

**O que olhar**

- `show ip interface brief`: todas as interfaces em uso `up/up`. O rótulo `SLARP` na coluna Method de uma interface série é apenas a marca do IOS para HDLC. A interface tem o endereço configurado à mão.
- `show ip ospf interface brief`: a coluna **Area** tem de bater com a planta (R6: Fa1/0 na 0, Fa1/1 na 1, Se2/0 na 2) e o custo tem de ser 10 nas FastEthernet.
- `show ip ospf neighbor`: o **Neighbor ID** tem de ser o router ID configurado (`2.0.0.n`), não um endereço de interface.
- `show ip ospf interface <if>`: se aparecer **`No Hellos (Passive interface)`**, a interface está passiva e não forma adjacência.

**Resultados observados no laboratório**

| Verificação | Resultado |
|---|---|
| R2, `show ip ospf` | Dois processos (`ospf 1` ID 1.0.0.2 e `ospf 2` ID 2.0.0.2), "autonomous system boundary router", redistribuição de `ospf 2` com métrica 100 e `subnets` |
| R2, `show ip route` | `192.168.1.0/25` via 192.168.1.129 `[110/20]`, e `192.168.1.0/24 is a summary ... Null0` |
| R3, `show ip route` | `O E2 192.168.1.0/24 [110/100] via 192.168.50.1`: a sumarização chega ao AS2 |
| R5, vizinhos | R3 `FULL/DR` e R4 `FULL/BDR` |
| R6, vizinhos | 2.0.0.3, 2.0.0.4 e 2.0.0.5 na Fa1/0 (todos `FULL`, com os papéis DR/BDR/DROTHER a depender da eleição), 2.0.0.7 na Fa1/1 (`FULL/BDR`) e 2.0.0.8 na Se2/0 (ponto a ponto, sem DR) |
| R7, vizinhos | 2.0.0.9 `FULL/DR` (Fa1/1) e 2.0.0.6 `FULL/DR` (Fa1/0) |
| R1, `show ip route` | Redes do AS2 como `O E2 [110/100]` via 192.168.1.130, e `192.168.1.0/24` também como `O E2` (ver nota na secção 6) |

**Teste de ponta a ponta**

- R1: `ping 172.16.1.2` com 80% de sucesso (o 1.º pacote perde-se na resolução ARP) e `trace 172.16.1.2` com o caminho 192.168.1.130 (R2), 192.168.50.2 (R3), 192.168.100.2 (R6), 172.20.1.2 (R7), 172.16.1.2.
- Terminal 1 (VPCS): `ping 172.16.1.2` com TTL 59 e `trace 172.16.1.2` com o mesmo caminho. A última linha, `Destination port unreachable`, é a forma como o VPCS termina o trace.

**Capturas de ecrã (outRes)**

![R1: configuração OSPF, interfaces na área 0 e running-config](evidencias/t2_r1_configuracao.png)

*R1: configuração OSPF, interfaces na área 0 e running-config*

![R2: interfaces up/up e `show ip ospf` com os dois processos](evidencias/t2_r2_interfaces.png)

*R2: interfaces up/up e `show ip ospf` com os dois processos*

![R2: processo `ospf 1` (ID 1.0.0.2), ASBR e redistribuição com métrica 100](evidencias/t2_r2_ospf_processos.png)

*R2: processo `ospf 1` (ID 1.0.0.2), ASBR e redistribuição com métrica 100*

![R2: área 0 do processo 2 com 2 interfaces](evidencias/t2_r2_ospf_area0.png)

*R2: área 0 do processo 2 com 2 interfaces*

![R2: vizinho R1 em FULL e rota sumarizada 192.168.1.0/24 para Null0](evidencias/t2_r2_vizinhos_rotas.png)

*R2: vizinho R1 em FULL e rota sumarizada 192.168.1.0/24 para Null0*

![R3: vizinho R2 em FULL e `O E2 192.168.1.0/24 [110/100]`](evidencias/t2_r3_vizinhos_rotas.png)

*R3: vizinho R2 em FULL e `O E2 192.168.1.0/24 [110/100]`*

![R4: interface Fa1/0 e vizinho R3](evidencias/t2_r4_vizinhos.png)

*R4: interface Fa1/0 e vizinho R3*

![R5: vizinhos R3 (DR) e R4 (BDR)](evidencias/t2_r5_vizinhos.png)

*R5: vizinhos R3 (DR) e R4 (BDR)*

![R6: network lines corretas, Fa1/0 na área 0, Fa1/1 na 1 e Se2/0 na 2](evidencias/t2_r6_config_corrigida.png)

*R6: network lines corretas, Fa1/0 na área 0, Fa1/1 na 1 e Se2/0 na 2*

![R7: interfaces na área 1 e vizinhos R9 e R6 em FULL](evidencias/t2_r7_vizinhos.png)

*R7: interfaces na área 1 e vizinhos R9 e R6 em FULL*

![R9: Fa1/0 na área 1, Fa1/1 e Fa2/0 na área 2](evidencias/t2_r9_interfaces.png)

*R9: Fa1/0 na área 1, Fa1/1 e Fa2/0 na área 2*

![R1: rotas O E2 do AS2, ping e trace para 172.16.1.2](evidencias/t2_r1_rotas_ping_trace.png)

*R1: rotas O E2 do AS2, ping e trace para 172.16.1.2*

![Terminal 1 (VPCS): ping e trace para 172.16.1.2](evidencias/t2_term1_ping_trace.png)

*Terminal 1 (VPCS): ping e trace para 172.16.1.2*


## 5. Erros encontrados e correções

### Erro 1: comandos rejeitados com `% Invalid input detected`

- **Sintoma:** `hostname R1` e `no ip domain lookup` devolveram `Invalid input` no prompt `Router>`.
- **Causa:** o prompt `Router>` é o modo EXEC de utilizador, onde os comandos de configuração não existem. Além disso, o comando é `no ip domain-lookup`, com hífen.
- **Correção:** `enable`, depois `configure terminal`, e só então os comandos de configuração. O prompt passa a `Router(config)#`.

![Erro 1: `Invalid input` no prompt `Router>`](evidencias/t2_erro1_modo_exec.png)

*Erro 1: `Invalid input` no prompt `Router>`*

### Erro 2: R6 com Fa1/0 na área errada e consola inutilizável

- **Sintoma:** a consola do R6 enchia-se de
  `%OSPF-4-ERRRCV: Received invalid packet: mismatch area ID, from backbone area must be virtual-link but not found from 192.168.100.x`,
  e não era possível escrever comandos.
- **Causa:** o `show running-config | section ospf` mostrava `network 192.168.100.0 0.0.0.255 area 1`. A interface Fa1/0 estava na **área 1**, enquanto R3, R4 e R5 enviam hellos da **área 0**. O R6 rejeitava-os por incompatibilidade de área.
- **Correção:**
  ```
  no logging console
  router ospf 2
   no network 192.168.100.0 0.0.0.255 area 1
   network 192.168.100.0 0.0.0.255 area 0
   network 172.20.1.0 0.0.0.3 area 1
   network 172.20.1.12 0.0.0.3 area 2
  ```
- **Lição:** o `no logging console` impede que as mensagens interrompam a escrita (os caracteres não se perdiam, o log só os intercalava). A linha errada tinha de ser removida com `no network ...`, porque o primeiro `network` que coincide com a interface decide a área.

![Erro 2: mensagens `mismatch area ID` na consola do R6](evidencias/t2_erro2_r6_mismatch_area_log.png)

*Erro 2: mensagens `mismatch area ID` na consola do R6*

![Erro 2: `network 192.168.100.0 0.0.0.255 area 1` e Fa1/0 na área 1](evidencias/t2_erro2_r6_config_area_errada.png)

*Erro 2: `network 192.168.100.0 0.0.0.255 area 1` e Fa1/0 na área 1*

### Erro 3: R5 aparecia como `192.168.100.254` em vez de `2.0.0.5`

- **Sintoma:** no `show ip ospf neighbor` do R6, o vizinho tinha o ID `192.168.100.254`.
- **Causa:** o `router-id 2.0.0.5` não estava em vigor. O IOS só aplica uma mudança de router ID depois de reiniciar o processo, e na falta dela usa o endereço mais alto de uma interface.
- **Correção no R5:**
  ```
  configure terminal
  router ospf 2
   router-id 2.0.0.5
   end
  clear ip ospf process
  write memory
  ```
  Respondeu-se `yes` à confirmação. A entrada antiga `192.168.100.254` desaparece do R6 em menos de um minuto, quando o dead timer expira.
- **Importância:** o enunciado exige router IDs `2.0.0.n` e as questões sobre LSAs olham para o campo *Advertising Router*.

![Erro 3: R6 vê o R5 como 192.168.100.254](evidencias/t2_erro3_r6_vizinhos_id_r5.png)

*Erro 3: R6 vê o R5 como 192.168.100.254*

![Erro 3: após o `clear ip ospf process`, o R5 aparece como 2.0.0.5 (a entrada antiga expira)](evidencias/t2_erro3_r6_vizinhos_apos_fix_r5.png)

*Erro 3: após o `clear ip ospf process`, o R5 aparece como 2.0.0.5 (a entrada antiga expira)*

### Erro 4: R8 sem vizinhos, embora os pings funcionassem

- **Sintoma:** R8 sem qualquer vizinho OSPF, e R6 e R9 sem ver o R8. Os pings entre R8, R9 e R6 tinham sucesso, logo as camadas 1 a 3 estavam corretas.
- **Causa:** `show ip ospf interface FastEthernet1/1` e `show ip ospf interface Serial2/0` mostravam **`No Hellos (Passive interface)`**. O `passive-interface default` estava ativo e faltavam as linhas `no passive-interface` das duas interfaces ligadas a routers.
- **Correção:**
  ```
  undebug all
  configure terminal
  router ospf 2
   no passive-interface Serial2/0
   no passive-interface FastEthernet1/1
   end
  write memory
  ```
  A Fa1/0 (nuvem 172.18.1.0/24) fica passiva.
- **Lição de diagnóstico:** separar primeiro o problema de IP (`ping` aos vizinhos) do problema de OSPF (`show ip ospf interface`, `debug ip ospf hello`). Uma interface passiva mantém o endereço anunciado, mas não envia hellos, por isso o ping funciona e a adjacência não.

![Erro 4: R8 sem vizinhos OSPF](evidencias/t2_erro4_r8_sem_vizinhos.png)

*Erro 4: R8 sem vizinhos OSPF*

![Erro 4: R9 só vê o R7, não vê o R8](evidencias/t2_erro4_r9_vizinhos.png)

*Erro 4: R9 só vê o R7, não vê o R8*

![Erro 4: `No Hellos (Passive interface)` na Fa1/1 e na Se2/0 do R8](evidencias/t2_erro4_r8_passive_hello.png)

*Erro 4: `No Hellos (Passive interface)` na Fa1/1 e na Se2/0 do R8*

### Diagnóstico enganador (falso alarme): `SLARP` na Se2/0 do R8

Por momentos suspeitou-se que o endereço série não estava aplicado. O `show running-config interface Serial2/0` mostrou `ip address 172.20.1.13 255.255.255.252`, ou seja, estava configurado. O rótulo `SLARP` é só a marca do IOS para HDLC, e a causa real era a interface passiva (erro 4).

![Falso alarme: endereço da Se2/0 configurado à mão, rotulado `SLARP`](evidencias/t2_erro4_r8_serial_slarp.png)

*Falso alarme: endereço da Se2/0 configurado à mão, rotulado `SLARP`*

### Boa prática final

Em cada router, comparar `show ip ospf interface brief` e `show running-config | section ospf` com o plano da secção 3. Os erros 2, 3 e 4 seriam todos detetados por esta verificação.

## 6. Observações sobre o comportamento esperado

- **`O E2 192.168.1.0/24` no R1:** o AS2 reanuncia a sumarização do R2 de volta para o AS1, porque a redistribuição é feita nos dois sentidos. É inofensivo: as rotas específicas (`/25` ligada e `/26` aprendidas) ganham por serem mais específicas. O enunciado não pede filtragem.
- **Rotas externas E2 com métrica 100:** a métrica de uma rota E2 não aumenta ao atravessar o AS, por isso aparece 100 em todos os routers.
- **Rede multi-acesso 192.168.100.0/24:** só o DR e o BDR ficam `FULL` com todos. Os restantes ficam em `2WAY` entre si (`DROTH`). Esta observação é relevante para a questão 5 do enunciado.
- **R9 sem ligação à área 0:** o tráfego entre as áreas 1 e 2 e a área 0 passa pelo R6. Este desenho é intencional, e explica a assimetria de caminhos pedida em algumas questões. Não deve ser "corrigido".
- **Ping inicial perdido (`.!!!!`):** é a resolução ARP do primeiro pacote.


## 7. Respostas às questões do enunciado

Cada questão segue a mesma ordem: **o que se pede**, **o conceito necessário** (para quem está a estudar), **a evidência** (capRes e outRes) e a **resposta final** (texRes). As respostas finais estão em blocos de citação, prontas a copiar.

> **Nota sobre as evidências.** As capturas de pacotes estão como capturas de ecrã do Wireshark e os outputs de consola como capturas de ecrã do terminal. Estão todas em `evidencias/` com o prefixo `t2_q<n>_`. Os ficheiros `.pcapng` não foram guardados nesta pasta. Se a entrega os exigir, exportá-los do GNS3 (pasta de capturas do projeto) e guardá-los ao lado.

### Mini-glossário (consultar quando for preciso)

#### Os cinco tipos de pacote OSPF

| Pacote | Para que serve | Quando aparece |
|---|---|---|
| **Hello** | Descobrir vizinhos, manter a adjacência viva (Hello 10 s, Dead 40 s), eleger DR/BDR e confirmar que a comunicação é nos dois sentidos (lista de vizinhos ouvidos) | Sempre, a cada 10 s |
| **DB Description (DBD)** | Enviar o **índice** da base de dados (só os cabeçalhos dos LSA) e escolher o master | Ao formar uma adjacência |
| **LS Request (LSR)** | Pedir ao vizinho os LSA completos que faltam ou estão desatualizados | Depois do DBD |
| **LS Update (LSU)** | Transportar LSA completos (é o pacote do *flooding*) | Na sincronização inicial e sempre que algo muda |
| **LS Acknowledge (LSAck)** | Confirmar a receção de LSA, o que torna o flooding fiável | A seguir a cada LSU |

#### Estados de um vizinho

`Down` (sem notícias) → `Init` (ouvi o Hello dele) → `2-Way` (ele também me ouviu: comunicação bidirecional) → `ExStart` (escolher o master) → `Exchange` (trocar DBD) → `Loading` (pedir LSA em falta) → `Full` (bases de dados iguais).

Numa rede com DR, só o DR e o BDR chegam a `Full` com todos. Os restantes ficam em `2-Way` entre si.

#### Os tipos de LSA que aparecem neste trabalho

| Tipo | Nome | Quem origina | Onde circula | O que descreve | Link State ID |
|---|---|---|---|---|---|
| 1 | Router-LSA | Todos os routers | Dentro da área | As ligações do router nessa área (tipo, ID, Data, custo) | Router ID de quem origina |
| 2 | Network-LSA | O DR | Dentro da área | Os routers ligados a uma rede com DR e a máscara | IP da interface do DR |
| 3 | Summary-LSA | O ABR | Para a área vizinha | Uma rede de **outra** área e o custo até lá | Endereço da rede |
| 4 | ASBR-Summary-LSA | O ABR | Para a área vizinha | Como chegar ao ASBR | Router ID do ASBR |
| 5 | AS-External-LSA | O ASBR | Todo o AS | Uma rede **fora** do AS (rota redistribuída) | Endereço da rede externa |

O **Advertising Router** é sempre o router que originou o LSA. O Link State ID, por sua vez, **depende do tipo** (última coluna).


### Questão 1: como dois routers sincronizam as bases de dados (R2 e R3)

**O que se pede.** Arrancar R3 a R9 e esperar; capturar a ligação R2-R3; arrancar o R2 e esperar que a adjacência com o R3 passe de `LOADING` a `FULL`; parar a captura. Depois analisar os DB Description (1a) e os LS Request (1b).

> **Conceito: o que acontece quando um router arranca.** Um router acabado de arrancar tem a base de dados de estados de ligação (LSDB) vazia e não consegue calcular rotas sem ela. A solução é copiá-la de um vizinho, mas sem a transferir às cegas. Os dois routers fazem três coisas por esta ordem:
>
> 1. **ExStart:** decidem quem conduz a conversa (o *master*).
> 2. **Exchange:** trocam um **índice** da base de dados (DB Description, só os cabeçalhos dos LSA).
> 3. **Loading:** cada um pede só o que lhe falta (LS Request) e recebe-o (LS Update).
>
> Quando não falta nada, a adjacência fica em **Full**.

#### capRes: antes de o R2 arrancar

Com o R3 sozinho na ligação, só se vê um Hello do R3 a cada 10 s, sem vizinhos listados (90 bytes).

![Captura R2-R3 antes do arranque do R2: só Hellos do R3](evidencias/t2_q1_baseline_hellos.png)

*Antes do R2: apenas Hello Packets do R3 (192.168.50.2) para 224.0.0.5, de 10 em 10 s.*

#### capRes: o arranque do R2

![Captura R2-R3: sincronização das bases de dados](evidencias/t2_q1_captura_r2_r3.png)

*Captura da ligação R2-R3 durante o arranque do R2 (frames 40 a 69).*

Como ler a captura:

| Frames | Pacote | Estado | O que se vê |
|---|---|---|---|
| 45 e 46 | DB Description (78 bytes) | ExStart | DBD iniciais **sem** cabeçalhos de LSA. Cada router propõe-se como master (o frame 46 está aberto mais abaixo) |
| 47 e 48 | DB Description (118 e 138 bytes) | Exchange | Já levam cabeçalhos de LSA: 78 bytes de base mais 20 por cabeçalho, ou seja, 2 cabeçalhos do R2 e 3 do R3 |
| 49, 51, 53 | DB Description (78 bytes) | Exchange | DBD sem cabeçalhos, a confirmar e a fechar a troca |
| 50 | **LS Request** (82 bytes) | Loading | O R2 pede dois LSA ao R3 |
| 52 | **LS Update** (158 bytes) | Loading | O R3 responde com esses dois LSA |
| 55 a 57, 59, 61, 63, 64 | LS Update e LS Acknowledge para 224.0.0.5 | Full | Já em adjacência completa, novos LSA são inundados e confirmados |

Repare que o Hello passa de 90 para **94 bytes** depois da adjacência: o campo *Active Neighbor* passa a listar o vizinho (4 bytes por vizinho).

#### 1a) DB Description (texRes)

O frame 46 mostra os campos de um DB Description:

![DB Description (frame 46)](evidencias/t2_q1_dbd_frame46.png)

*DBD do R3: I (Init) = 1, M (More) = 1, MS (Master) = 1, DD Sequence 9168, MTU 1500.*

- **I (Init):** é o primeiro DBD da troca.
- **M (More):** ainda há mais DBD a enviar.
- **MS (Master/Slave):** este router propõe-se como master. O **maior Router ID** ganha, e neste caso é o R3 (2.0.0.3 contra 2.0.0.2).
- **DD Sequence:** numera os DBD. O master define-o, o slave responde com o mesmo número, e assim cada DBD é confirmado.
- **MTU:** os dois lados têm de coincidir, senão a adjacência fica presa em ExStart.

> **1a (texRes).** Os DB Description servem para **comparar as bases de dados** de dois vizinhos sem as transferir inteiras. Cada DBD contém apenas os **cabeçalhos dos LSA** (tipo, Link State ID, Advertising Router, número de sequência, checksum e idade), isto é, um índice da base de dados, e serve também para negociar quem é o **master** (maior Router ID; os bits I, M e MS e o DD Sequence ordenam a troca). Com os índices, cada router descobre que LSA lhe faltam ou que o vizinho tem mais recentes, e pede-os depois com LS Request. É preciso trocá-los sempre que se forma uma nova adjacência porque a base de dados tem de ser **idêntica em todos os routers da área** para que o SPF dê o mesmo grafo a todos, e o flooding normal só propaga alterações futuras. O que se perdeu enquanto o router esteve parado, ou que ele nunca viu, só se recupera com esta sincronização inicial. Aqui o R2 tinha acabado de arrancar com a base de dados vazia, e o R3 já tinha a informação da área 0 do AS2.

#### 1b) LS Request (texRes)

![LS Request (frame 50)](evidencias/t2_q1_b_lsrequest.png)

*O R2 (192.168.50.1) pede ao R3 dois Router-LSA: o de 2.0.0.3, 2.0.0.2, 2.0.0.5, 2.0.0.4 e 2.0.0.6*

Um pedido não leva o conteúdo do LSA, só a sua **identificação**: o trio (tipo, Link State ID, Advertising Router).

![LS Update em resposta (frame 52)](evidencias/t2_q1_lsupdate_frame52.png)

*A resposta (frame 52): o LS Update traz os dois LSA pedidos. O primeiro é o Router-LSA do R3 (idade 150 s, seq 0x80000003, 3 ligações Stub).*

Dois detalhes interessantes:

1. O R2 pede o Router-LSA do **próprio R2** (2.0.0.2). A explicação mais provável é que o R3 ainda guardava uma cópia antiga desse LSA, de antes de o R2 reiniciar. Ao recebê-la, o R2 fica a saber o número de sequência que tinha e passa a originar o seu LSA com um número maior.
2. Só o R2 enviou LS Request nesta captura. O R3 não precisou: nenhum cabeçalho anunciado pelo R2 era novo ou mais recente do que o que o R3 já tinha.

> **Observação.** O Router-LSA do R3 trocado aqui ainda só descreve ligações **Stub** (192.168.100.0/24, 192.168.60.0/24 e 192.168.50.0/24, custo 10 cada). Não há ligação *Transit*, o que é coerente com o R3 ainda não ter adjacências completas na 192.168.100.0 quando esse LSA foi originado, e com a base de dados pequena que se vê trocada.

> **1b (texRes).** Os LS Request servem para **pedir ao vizinho os LSA completos** que faltam ou que o vizinho tem numa versão mais recente. Cada pedido identifica o LSA pelo trio (tipo, Link State ID, Advertising Router), sem conteúdo. Cada router sabe o que tem de pedir **comparando** os cabeçalhos recebidos nos DBD com a sua própria base de dados: se não tem aquele LSA, ou se o do vizinho é mais recente (comparam-se o número de sequência, depois o checksum e depois a idade), acrescenta-o à sua lista de pedidos. Na captura, o R2 pede ao R3 o Router-LSA do R3 (que desconhecia) e uma cópia antiga do seu próprio Router-LSA, e o R3 responde com um LS Update que traz esses dois LSA. O R3 não enviou LS Request porque o R2 não lhe anunciou nada de novo.


### Questão 2: o que o R6 anuncia em cada área

**O que se pede.** No R6, correr `show ip ospf database router 2.0.0.6`; identificar as ligações anunciadas em cada área (2a) e compará-las nas áreas 1 e 2, relacionando-as com o tipo de ligação do R6 (2b).

> **Conceito: um Router-LSA por área.** O Router-LSA descreve, para **uma área**, as ligações do router **nessa área**. O R6 tem interfaces em três áreas (é ABR), por isso origina três Router-LSA, cada um só com as suas interfaces dessa área, e liga em todos o bit **B** (*Area Border Router*).
>
> Cada ligação tem um tipo, um ID, um Data e uma métrica:
>
> | Tipo | Código | Link ID | Link Data | Quando aparece |
> |---|---|---|---|---|
> | Point-to-point | 1 | Router ID do vizinho | IP da interface local | Ligações série ou ponto a ponto |
> | Transit | 2 | IP da interface do **DR** | IP da interface local | Redes com DR (Ethernet) com vizinhos |
> | Stub | 3 | Endereço da rede | Máscara | Redes sem vizinhos, ou a sub-rede de uma ligação ponto a ponto |

#### outRes

![Router-LSA do R6 na área 0](evidencias/t2_q2_r6_router_lsa_area0.png)

*Área 0: uma ligação Transit para o DR 192.168.100.1.*

![Router-LSA do R6 nas áreas 1 e 2](evidencias/t2_q2_r6_router_lsa_area1_area2.png)

*Áreas 1 e 2: uma ligação Transit na área 1 e uma ligação ponto a ponto (mais o stub) na área 2.*

#### 2a) Ligações que o R6 anuncia (texRes)

| Área | Interface | Tipo | Link ID | Link Data | Métrica |
|---|---|---|---|---|---|
| 0 | Fa1/0 (192.168.100.2) | Transit | 192.168.100.1 (DR, o R3) | 192.168.100.2 | 10 |
| 1 | Fa1/1 (172.20.1.1) | Transit | 172.20.1.1 (DR, o próprio R6) | 172.20.1.1 | 10 |
| 2 | Se2/0 (172.20.1.14) | Point-to-point | 2.0.0.8 (o R8) | 172.20.1.14 | 647 |
| 2 | Se2/0 | Stub | 172.20.1.12 | 255.255.255.252 | 647 |

O número de ligações é 1 (área 0), 1 (área 1) e 2 (área 2). A segunda ligação da área 2 não está visível no print, mas decorre de o LSA ter 2 ligações e 48 bytes de comprimento, e é igual ao stub do Router-LSA do R8 (questão 3f).

> **2a (texRes).** Na **área 0** o R6 anuncia uma ligação *Transit Network* (Fa1/0), com Link ID 192.168.100.1 (o endereço do DR, o R3), Link Data 192.168.100.2 (a sua própria interface) e métrica 10. Na **área 1** anuncia uma ligação *Transit Network* (Fa1/1), com Link ID e Link Data 172.20.1.1 e métrica 10. Na **área 2** anuncia uma ligação *point-to-point* para o R8 (Link ID 2.0.0.8, Link Data 172.20.1.14, métrica 647) e uma ligação *Stub* para a rede 172.20.1.12/30 (métrica 647). Nos três Router-LSA o R6 assinala-se como *Area Border Router*.

#### 2b) Comparação entre as áreas 1 e 2 (texRes)

As duas áreas têm **uma interface** cada, mas o número de ligações anunciadas é diferente (1 contra 2), e o motivo é o tipo de ligação:

| | Área 1 (Fa1/1, Ethernet) | Área 2 (Se2/0, série) |
|---|---|---|
| Tipo | Rede de difusão, com DR | Ponto a ponto |
| Ligação no Router-LSA | **Transit** | **Point-to-point** + **Stub** |
| Link ID | IP do DR da rede | Router ID do vizinho (2.0.0.8) |
| O vizinho aparece nomeado? | Não | Sim |
| A sub-rede aparece no Router-LSA? | Não, está no Network-LSA do DR | Sim, numa ligação Stub à parte |
| Métrica | 10 (Fast Ethernet) | 647 (série de 1,544 Mb/s) |

As métricas vêm de `auto-cost reference-bandwidth 1000`: 1000 Mb/s dividido por 100 Mb/s dá 10, e 1000 dividido por 1,544 dá cerca de 647.

> **2b (texRes).** Na área 1 a interface é Ethernet, uma rede com DR, e o R6 anuncia **uma única ligação Transit**: o grafo trata a rede como um nó (o DR), e o vizinho (R7) não aparece nomeado no Router-LSA, mas no Network-LSA que o DR origina; a sub-rede também não vem no Router-LSA. Na área 2 a interface é série e a ligação é **ponto a ponto**, entre dois routers apenas e sem DR, por isso o Router-LSA nomeia diretamente o vizinho (Link ID = Router ID 2.0.0.8, Link Data = IP local) e acrescenta uma **ligação Stub** para descrever a sub-rede 172.20.1.12/30, daí as duas ligações. Na área 1 o Link ID e o Link Data coincidem (172.20.1.1) porque o próprio R6 era o DR dessa rede nesse momento. O DR de uma rede depende da ordem de arranque dos routers, como se vê nas questões 4 e 5.


### Questão 3: um vizinho desaparece e volta (R8 e R9)

**O que se pede.** Parar o R7 e esperar; no R8 ativar `debug ip ospf adj`; iniciar uma captura na ligação R6-R8 e parar o R9 durante uns minutos. Depois analisar o que acontece ao vizinho R9 (3a), os LSA que o R8 envia ao R6 (3b), o regresso do R9 (3c a 3f) e a estabilização final com o R7 de volta (3g).

> **Conceito: como o OSPF deteta uma falha.** Não há nenhuma mensagem do tipo "o R9 morreu". Cada router espera um Hello do vizinho a cada 10 s. Se passar o **Dead interval (40 s)** sem nenhum Hello, o vizinho é dado como inativo (evento *InactivityTimer*). Só então o router reage, **reoriginando os seus próprios LSA** (com número de sequência maior) e inundando-os.

#### 3a) O que aconteceu ao R9 no R8 (outRes + texRes)

![Debug de adjacências no R8](evidencias/t2_q3a_r8_debug_adj.png)

*`debug ip ospf adj` no R8: o R9 passa de FULL a DOWN às 15:28:25.*

Como ler o debug:

| Linha | O que significa |
|---|---|
| `2.0.0.9 address 172.20.1.9 on FastEthernet1/1 is dead` | O Dead timer (40 s) expirou |
| `Nbr 2.0.0.9 ... from FULL to DOWN, Neighbor Down: Dead timer expired` | Mudança de estado do vizinho e o seu motivo |
| `Neighbor change Event` e `DR/BDR election on FastEthernet1/1` | Mudar um vizinho obriga a repetir a eleição do DR |
| `Elect BDR 0.0.0.0` e `Elect DR 2.0.0.8` / `DR: 2.0.0.8 (Id) BDR: none` | O R8 fica DR sozinho, sem BDR |
| `Build router LSA for area 2, router ID 2.0.0.8, seq 0x80000005` | O R8 reoriginou o seu Router-LSA |
| `No full nbrs to build Net Lsa for interface FastEthernet1/1` | Sem vizinhos Full, o DR já não pode manter o Network-LSA do segmento |

> **3a (texRes).** O estado do R9 no R8 passou de **FULL para DOWN** às 15:28:25. O evento que desencadeou a mudança foi o **InactivityTimer** (`Neighbor Down: Dead timer expired`): ao fim de 40 s sem receber Hellos do R9, o R8 considerou-o inativo. Nenhum pacote anunciou a falha, o OSPF deteta-a pela ausência de Hellos. A mudança de vizinho levou a uma nova eleição de DR/BDR na Fa1/1 (o R8 ficou DR sozinho, sem BDR) e à reoriginação do Router-LSA do R8 (seq 0x80000005).


#### 3b) LSA enviados pelo R8 ao R6 (capRes + texRes)

![Captura R6-R8](evidencias/t2_q3b_captura_r6_r8.png)

*Captura da ligação R6-R8. R8 = 172.20.1.13, R6 = 172.20.1.14. Os SLARP e CDP são ruído da ligação série.*

Logo a seguir ao Dead timer, o R8 envia **dois LS Update seguidos** e o R6 confirma com um LS Acknowledge:

| Tempo | Origem | Pacote | Bytes | Conteúdo |
|---|---|---|---|---|
| 60.168 | R8 | LS Update | 124 | **Router-LSA** do R8 (seq 0x80000005) |
| 60.198 | R8 | LS Update | 84 | **Network-LSA** do segmento 172.20.1.8/30, com idade máxima (MaxAge, 3600 s) |
| 62.687 | R6 | LS Acknowledge | 88 | Confirma os dois LSA (dois cabeçalhos de 20 bytes) |

> **Como sabemos o conteúdo só pelo tamanho.** Numa ligação série o cabeçalho HDLC tem 4 bytes. Um LS Update tem 4 (HDLC) + 20 (IP) + 24 (OSPF) + 4 (nº de LSA) = **52 bytes** antes dos LSA.
>
> - 124 − 52 = **72 bytes**: é o tamanho de um Router-LSA com 4 ligações (20 de cabeçalho + 4 + 4 × 12).
> - 84 − 52 = **32 bytes**: é o tamanho de um Network-LSA com 2 routers (20 + 4 de máscara + 2 × 4). É o mesmo comprimento do Network-LSA que se vê na questão 4.
> - O LS Ack tem 4 + 20 + 24 = 48 bytes fixos mais 20 por LSA confirmado: 88 − 48 = 40, ou seja, dois LSA.

**Qual a diferença entre os dois LS Update?** O primeiro **atualiza** um LSA (o Router-LSA, que continua válido). O segundo **retira** um LSA (o Network-LSA, que deixa de fazer sentido).

> **3b (texRes).** O R8 enviou ao R6 dois LS Update.
>
> **1) Router-LSA do R8 (124 bytes, seq 0x80000005).** É o LSA em que o R8 descreve as suas ligações na área 2. Com o R9 perdido, a ligação da Fa1/1 deixou de ser uma ligação *Transit* (já não há vizinhos nem DR efetivo) e passou a *Stub* para a rede 172.20.1.8/30. As restantes ligações mantêm-se: ponto a ponto para o R6 (172.20.1.13, custo 647), stub 172.20.1.12/30 e stub 172.18.1.0/24 (LAN do R8). Serve para o R6, e por seu intermédio o resto da área 2 e as outras áreas, saber que o R9 já não é alcançável através do R8.
>
> **2) Network-LSA do segmento 172.20.1.8/30 (84 bytes, idade máxima).** Era originado pelo R8 por ser o DR dessa rede. Sem vizinhos Full, o R8 deixa de poder representar o segmento e inunda o mesmo LSA com idade 3600 s (MaxAge), o que manda **apagá-lo** das bases de dados. O debug confirma-o (`No full nbrs to build Net Lsa`).
>
> A diferença é, portanto, que o primeiro LS Update atualiza um LSA que continua a existir e o segundo retira um LSA que deixou de ter razão de ser. O R6 responde com um **LS Acknowledge** (88 bytes) que confirma os dois, o que fecha o envio fiável.


#### 3c) A mensagem que prova a comunicação bidirecional (capRes + texRes)

Com a captura na ligação R8-R9 e o R9 a arrancar, a mensagem pedida é um **Hello do R8 cuja lista *Active Neighbor* contém 2.0.0.9**.

![Hello do R8 com o R9 na lista de vizinhos](evidencias/t2_q3c_hello_r8.png)

*Hello do R8 (Hello 10 s, Dead 40 s, prioridade 1) com `Active Neighbor: 2.0.0.9`.*

> **3c (texRes).** A mensagem é o **Hello enviado pelo R8** que traz o Router ID do R9 (2.0.0.9) no campo **Active Neighbor**. Esse campo lista os vizinhos de quem os routers já ouviram Hellos nos últimos 40 s. Quando o R9 recebe um Hello do R8 com o seu próprio Router ID lá dentro, fica a saber que **o R8 o ouviu**, e como ele já ouvia o R8 a comunicação é **nos dois sentidos**. É o evento *2-WayReceived*, que leva o R9 do estado Init a 2-Way e, a seguir, à eleição de DR/BDR e ao ExStart. Justifica-se a escolha porque o Hello é o único pacote que carrega a lista de vizinhos ouvidos: os DBD, os LSR e os LSU só começam depois do 2-Way, e a mera receção de um Hello do R8 só prova que o R9 ouve o R8, não que o R8 ouve o R9. Neste Hello já se vê a eleição feita: o R8 é DR (172.20.1.10) e o R9 é BDR (172.20.1.9). A primeira vez que o R8 lista o R9 é a que dispara o 2-Way.


#### 3d) Os anúncios trocados entre R8 e R9 (capRes + texRes)

Para a análise usámos o LS Update enviado pelo R8 ao R9 durante a sincronização (frame 24, 406 bytes), que transporta **9 LSA**:

![LS Update com 9 LSA (frame 24)](evidencias/t2_q3d_lsupdate_9_lsa.png)

*LS Update de 172.20.1.10 (R8) para 172.20.1.9 (R9): 3 Router-LSA, 4 Summary-LSA, 1 ASBR-Summary-LSA e 1 AS-External-LSA.*

> **Porquê tantos LSA de uma vez.** O R9 acabou de arrancar e tem a base de dados vazia. Passou pelo ExStart e Exchange e pediu ao R8 tudo o que lhe falta. O R8 responde com um LS Update que reúne a **base de dados da área 2** (e é só da área 2: o R9 não vê os Router-LSA das outras áreas).

| Tipo | Qtd. | Tamanho | Para que serve |
|---|---|---|---|
| **1** Router-LSA | 3 | 48, 72 e 48 bytes | Cada router da área 2 descreve as suas ligações e custos. É a base do grafo da área |
| **3** Summary-LSA | 4 | 28 bytes cada | O ABR (R6) anuncia à área 2 as redes que estão **noutras áreas**, com o custo desde o ABR, sem revelar a topologia dessas áreas |
| **4** ASBR-Summary-LSA | 1 | 28 bytes | O R6 diz como chegar ao ASBR (R2), que está na área 0, fora da área 2 |
| **5** AS-External-LSA | 1 | 36 bytes | O R2 (ASBR) anuncia uma rede **de fora do AS** (a 192.168.1.0/24, vinda do AS1) com métrica 100 |

Os três Router-LSA são os dos routers da área 2: o do **R6** (ABR, 2 ligações, 48 bytes, questão 2), o do **R8** (4 ligações, 72 bytes, questão 3f) e um terceiro de 48 bytes e 2 ligações, que será o do próprio **R9** (a sua cópia de antes de reiniciar, que o R9 pediu e que depois reoriginou com número de sequência maior).

> **3d (texRes).** No LS Update analisado (R8 para R9) encontrámos **quatro tipos** de LSA:
>
> - **Tipo 1, Router-LSA (3):** descrevem as ligações de cada router da área 2 (R6, R8 e R9). Permitem a cada router construir o grafo da área e correr o SPF.
> - **Tipo 3, Summary-LSA (4):** originados pelo ABR R6, anunciam à área 2 as redes das outras áreas (área 0 e área 1) com o respetivo custo. Dão a conhecer destinos fora da área sem expor a sua topologia interna.
> - **Tipo 4, ASBR-Summary-LSA (1):** originado pelo ABR R6, indica o custo para alcançar o ASBR (R2). Os routers da área 2 não conhecem o R2, que está noutra área, e precisam de saber como lá chegar para usar os LSA de tipo 5.
> - **Tipo 5, AS-External-LSA (1):** originado pelo ASBR R2, anuncia a rede externa 192.168.1.0/24 (redistribuída do AS1, sumarizada). Circula em todo o AS, exceto em áreas stub.
>
> O Network-LSA (tipo 2) do segmento R8-R9 também existe na base de dados da área 2 (questão 4), mas não vem neste LS Update.


#### 3e) Campos Link State ID e Advertising Router, por tipo (texRes)

A pergunta pede, para cada tipo, o significado destes dois campos e de outros campos importantes, com exemplos concretos.

> **Regra geral.** O **Advertising Router** diz sempre **quem originou** o LSA (o seu Router ID). O **Link State ID** diz **o que o LSA descreve**, e o seu significado muda com o tipo. O par (tipo, Link State ID, Advertising Router) identifica o LSA, e o **número de sequência** (mais alto = mais recente), a **idade** e o **checksum** servem para saber qual é a cópia mais recente.

| Tipo | Link State ID | Advertising Router | Outros campos importantes |
|---|---|---|---|
| **1** Router | Router ID de quem origina | O mesmo router | Flags (bit **B** = ABR, bit **E** = ASBR); número de ligações; por cada ligação: tipo, **Link ID**, **Link Data** e **métrica** |
| **2** Network | IP da interface do **DR** na rede | O DR | Máscara da rede e lista de **Attached Routers** (os routers ligados ao segmento) |
| **3** Summary | Endereço da **rede** anunciada | O ABR | Máscara da rede e **métrica** (custo desde o ABR até à rede) |
| **4** ASBR-Summary | **Router ID do ASBR** | O ABR | **Métrica** (custo desde o ABR até ao ASBR) |
| **5** AS-External | Endereço da **rede externa** | O ASBR | Máscara, **tipo de métrica** (bit E: E2 quando ligado), **métrica**, **Forwarding Address** (0.0.0.0 = enviar para o ASBR) e *route tag* |

**Exemplos concretos neste trabalho**

- **Tipo 1.** Router-LSA do R8: Link State ID **2.0.0.8**, Advertising Router **2.0.0.8**, seq 0x80000006, 4 ligações (detalhadas na 3f).
- **Tipo 2.** Network-LSA do segmento R8-R9 (questão 4): Link State ID **172.20.1.10** (a interface do R8, que é o DR), Advertising Router **2.0.0.8**, máscara /30, Attached Routers **2.0.0.8** e **2.0.0.9**.
- **Tipo 3.** Summary-LSA originado pelo R6 (Advertising Router **2.0.0.6**) para uma rede de outra área, por exemplo Link State ID **192.168.100.0**, máscara /24, métrica igual ao custo do R6 até àquela rede (10).
- **Tipo 4.** ASBR-Summary-LSA do R6 para o R2: Link State ID **2.0.0.2**, Advertising Router **2.0.0.6**, métrica igual ao custo do R6 até ao R2.
- **Tipo 5.** AS-External-LSA do R2: Link State ID **192.168.1.0**, máscara /24, Advertising Router **2.0.0.2**, métrica **100** do tipo E2 e Forwarding Address 0.0.0.0.

> *Nota.* Os valores concretos dos quatro Summary-LSA (redes e métricas) e do LSA de tipo 4 e 5 são os que a topologia e os custos implicam. Podem ser conferidos abrindo cada LSA do frame 24 no Wireshark.

> **3e (texRes).** Em todos os tipos, o **Advertising Router** é o Router ID do router que originou o LSA. O **Link State ID** depende do tipo: no **Router-LSA** é o Router ID de quem o origina (2.0.0.8 no LSA do R8); no **Network-LSA** é o endereço da interface do **DR** (172.20.1.10); no **Summary-LSA** é o endereço da rede de destino anunciada pelo ABR (por exemplo 192.168.100.0 anunciada pelo R6); no **ASBR-Summary-LSA** é o Router ID do ASBR (2.0.0.2); e no **AS-External-LSA** é o endereço da rede externa (192.168.1.0, originada pelo ASBR 2.0.0.2). Os outros campos relevantes para construir o grafo são: no Router-LSA, as flags B e E, o número de ligações e, por ligação, tipo, ID, Data e métrica; no Network-LSA, a máscara e a lista de routers ligados (que liga o nó rede aos routers); no Summary-LSA e no ASBR-Summary-LSA, a métrica desde o ABR; e no AS-External-LSA, a métrica E2 (100), o Forwarding Address (0.0.0.0) e a máscara. O número de sequência, a idade e o checksum são comuns a todos e servem para distinguir a cópia mais recente do LSA.


#### 3f) O último Router-LSA do R8 (capRes + texRes)

![Último Router-LSA originado pelo R8](evidencias/t2_q3f_r8_router_lsa.png)

*Router-LSA do R8: Link State ID e Advertising Router 2.0.0.8, seq 0x80000006, idade 5 s, 4 ligações.*

O seq 0x80000006 é o seguinte ao 0x80000005 que vimos na 3a, e tem a ligação da Fa1/1 de novo como *Transit*, porque o R9 voltou.

| # | Tipo | ID | Data | Métrica | Significado |
|---|---|---|---|---|---|
| 1 | **Point-to-point** | **2.0.0.6** | 172.20.1.13 | 647 | Ligação série ao R6. **ID** = Router ID do vizinho (R6). **Data** = IP da interface local do R8 (Se2/0) |
| 2 | **Stub** | **172.20.1.12** | 255.255.255.252 | 647 | A sub-rede da ligação série. **ID** = endereço da rede, **Data** = máscara (/30) |
| 3 | **Transit** | **172.20.1.10** | 172.20.1.10 | 10 | Ligação à rede com o R9 (Fa1/1). **ID** = IP da interface do **DR** (o próprio R8). **Data** = IP local do R8 (coincidem por o R8 ser o DR) |
| 4 | **Stub** | **172.18.1.0** | 255.255.255.0 | 10 | LAN do R8 (Fa1/0, passiva, sem vizinhos). **ID** = endereço da rede, **Data** = máscara (/24) |

> **3f (texRes).** O último Router-LSA originado pelo R8 (seq 0x80000006) tem 4 ligações. (1) **Point-to-point** para o R6: ID = 2.0.0.6 (Router ID do vizinho), Data = 172.20.1.13 (IP da interface local), custo 647. (2) **Stub** 172.20.1.12/30: ID = endereço da rede, Data = máscara 255.255.255.252, custo 647, que descreve a sub-rede da ligação ponto a ponto. (3) **Transit** para a rede com o R9: ID = 172.20.1.10 (endereço da interface do DR, que é o próprio R8) e Data = 172.20.1.10 (IP local), custo 10. (4) **Stub** 172.18.1.0/24: ID = endereço da rede, Data = máscara 255.255.255.0, custo 10, que é a LAN do R8, anunciada como stub por não ter outros routers. Em ligações ponto a ponto e Transit o ID identifica o vizinho (ou o DR) e o Data o IP local, e nas Stub o ID é a rede e o Data a máscara.


#### 3g) Depois de o R7 voltar (capRes + texRes)

Com o R7 de novo ligado e a rede estabilizada, o que o R8 envia ao R9 resume-se a Hellos, LS Update e LS Acknowledge:

![Captura R8-R9 depois de o R7 voltar](evidencias/t2_q3g_captura_r8_r9.png)

*R8 = 172.20.1.10, R9 = 172.20.1.9. Os LS Update do R8 têm todos 90 bytes e os LS Acknowledge do R9 têm 78 ou 98 bytes.*

| Tempo | Origem | Pacote | Bytes | LSA transportados |
|---|---|---|---|---|
| 36.106 | R8 | LS Update | 90 | 1 |
| 38.616 | R9 | LS Acknowledge | 78 | confirma 1 |
| 41.100 | R8 | LS Update | 90 | 1 |
| 43.610 | R9 | LS Acknowledge | 78 | confirma 1 |
| 46.113 e 46.153 | R8 | 2 LS Update | 90 cada | 1 cada |
| 48.625 | R9 | LS Acknowledge | 98 | confirma 2 |

> **Como sabemos o conteúdo pelos tamanhos.** Num segmento Ethernet o LS Update tem 14 + 20 + 24 + 4 = **62 bytes** antes dos LSA, logo 90 − 62 = **28 bytes por LSA**, que é o tamanho de um **Summary-LSA**. O LS Ack tem 14 + 20 + 24 = 58 bytes mais 20 por cabeçalho: 78 confirma 1 LSA e 98 confirma 2.

Ou seja, o R8 enviou **quatro Summary-LSA** (tipo 3), cada um no seu LS Update, e o R9 confirmou-os. O R9 não enviou LSA nenhum.

**Porquê Summary-LSA e nada mais?** O arranque do R7 muda a **área 1** (novos Router-LSA do R7, do R6 e do R9 nessa área). Esses LSA de tipo 1 e 2 **ficam dentro da área 1**, e o R8, que está na área 2, não os recebe. Quem passa a novidade para a área 2 é o ABR R6, que a **resume**: as redes da área 1 que voltaram a ser alcançáveis (por exemplo 172.20.1.4/30 e 172.16.1.0/24, a LAN do Terminal 2) são anunciadas à área 2 em Summary-LSA, com Advertising Router 2.0.0.6. O R8 recebe-os do R6 e reenvia-os ao R9. O R8 não originou nenhum Router-LSA novo porque as suas ligações não mudaram.

> *Nota.* Os valores exatos (Link State ID e métrica) dos quatro Summary-LSA devem ser conferidos abrindo cada LS Update no Wireshark. O que a captura prova sem ambiguidade é o tipo (28 bytes por LSA) e o número (4).

> **3g (texRes).** Depois do arranque do R7, o R8 enviou ao R9 **quatro LSA do tipo Summary (tipo 3)**, cada um num LS Update, e o R9 respondeu com LS Acknowledge. Foram originados pelo ABR R6 (Advertising Router 2.0.0.6) e têm como finalidade **informar a área 2 das redes da área 1 que voltaram a ficar alcançáveis** com o R7 (como 172.20.1.4/30 e a LAN 172.16.1.0/24), indicando prefixo e custo calculado pelo ABR, sem expor a topologia interna da área 1. Não houve Router-LSA nem Network-LSA porque estes não saem da área onde são originados, e as ligações do R8 não mudaram. Os LS Acknowledge do R9 são a confirmação do OSPF: um LSA é retransmitido até ser confirmado. Como o R9 não tem ligação à área 0, não resume nada para a área 2, e é o R6 que cumpre essa função (consequência que é decisiva nas questões 6 e 7).


### Questão 4: DR e Network-LSA no R8

**O que se pede.** Em cada interface do R8 correr `show ip ospf interface` e ver se foi eleito um DR (4a); ver se há Network-LSA nas ligações com DR e em que situação é gerado (4b); relacionar o originador do Network-LSA com o DR (4c).

> **Conceito: DR e Network-LSA.** Numa rede de difusão (Ethernet) com vários routers, em vez de cada um se ligar a todos os outros, elege-se um **DR** (Designated Router) e um **BDR** (reserva). Todos os outros falam só com eles. A eleição escolhe a **maior prioridade** e, em empate, o **maior Router ID**, mas **não é preemptiva**: quem já é DR não perde o lugar quando chega um router com ID maior. O DR representa a rede no grafo com um **Network-LSA**. Nas ligações ponto a ponto não há DR.

#### outRes

![R8: Fa1/0 e Se2/0](evidencias/t2_q4_r8_fa10_serial.png)

*Fa1/0 (LAN passiva): State DR, sem BDR, 0 vizinhos. Se2/0 (série): POINT_TO_POINT, adjacente com 2.0.0.6.*

![R8: Fa1/1](evidencias/t2_q4_r8_fa11.png)

*Fa1/1 (para o R9): o R8 é DR (2.0.0.8) e o R9 é BDR (2.0.0.9). Adjacência Full com o BDR.*

![R8: vizinhos e Network-LSA](evidencias/t2_q4_r8_vizinhos_network_lsa.png)

*`show ip ospf neighbor` e `show ip ospf database network`.*

#### 4a) Em que ligações foi ou não eleito um DR (texRes)

| Interface | Ligação | Tipo de rede | DR/BDR | O que se vê |
|---|---|---|---|---|
| Fa1/1 | R8 e R9 (172.20.1.8/30) | BROADCAST | **Sim** | DR = 2.0.0.8 (R8), BDR = 2.0.0.9 (R9) |
| Fa1/0 | LAN 172.18.1.0/24 | BROADCAST (passiva) | Só o R8 | State DR, "No backup designated router", "No Hellos (Passive interface)", 0 vizinhos |
| Se2/0 | R8 e R6 (172.20.1.12/30) | POINT_TO_POINT | **Não** | State POINT_TO_POINT, sem campo DR/BDR; no `show ip ospf neighbor` aparece `FULL/ -` |

O R8 foi eleito DR na Fa1/1 apesar de ter o ID **menor** (2.0.0.8 contra 2.0.0.9): ambos têm prioridade 1 e a eleição não é preemptiva. O R8 já era DR sozinho na rede quando o R9 voltou (questão 3), e manteve o papel.

> **4a (texRes).** Há eleição de DR/BDR apenas nas ligações de **difusão** com outros routers. Na **Fa1/1** (rede com o R9) foi eleito DR o R8 (2.0.0.8) e BDR o R9 (2.0.0.9). Ambos têm prioridade 1, e o R9 tem o ID maior, mas a eleição **não é preemptiva**: o R8 já estava ativo como DR quando o R9 chegou e manteve o papel. Na **Fa1/0** (LAN passiva), o R8 é o único router da rede, elege-se DR sem BDR e não tem vizinhos. Na **Se2/0** (série para o R6) a ligação é **ponto a ponto**, onde não se faz eleição porque só há dois routers e basta uma adjacência entre eles, e a adjacência com o R6 fica Full sem papel de DR ou BDR.

#### 4b) O Network-LSA (texRes)

O `show ip ospf database network` mostra um único Network-LSA na área 2:

| Campo | Valor |
|---|---|
| Link State ID | 172.20.1.10 (endereço do DR) |
| Advertising Router | 2.0.0.8 |
| Máscara | /30 |
| Attached Routers | 2.0.0.8 e 2.0.0.9 |
| Número de sequência | 0x80000002 |
| Comprimento | 32 bytes |

Não existe Network-LSA para a ligação ao R6 (ponto a ponto, sem DR) nem para a LAN da Fa1/0 (o R8 é DR mas está sozinho). A condição aparece no debug da 3a: `No full nbrs to build Net Lsa`.

> **4b (texRes).** O Network-LSA só se gera nas ligações **com DR** e **apenas quando o DR tem pelo menos um vizinho em estado Full** nessa rede. Na Fa1/1 existe (Link State ID 172.20.1.10, Advertising Router 2.0.0.8, máscara /30, routers ligados 2.0.0.8 e 2.0.0.9). Não existe na Se2/0, por ser ponto a ponto e não ter DR, nem na Fa1/0, onde o R8 é DR mas não tem vizinhos. Na questão 3a viu-se o oposto: quando o R9 caiu, o R8 ficou sem vizinhos Full na Fa1/1 e retirou o Network-LSA (`No full nbrs to build Net Lsa`).

#### 4c) Relação entre o originador do Network-LSA e o DR (texRes)

> **4c (texRes).** O Network-LSA é originado **exclusivamente pelo DR** da rede a que se refere. O Advertising Router do LSA é o Router ID do DR (2.0.0.8) e o Link State ID é o endereço da interface do DR nessa rede (172.20.1.10). Os restantes routers não originam Network-LSA: descrevem a rede no seu Router-LSA com uma ligação *Transit* cujo Link ID aponta para o endereço do DR. Se o DR mudar, o LSA deixa de ser originado pelo antigo e passa a ser originado pelo novo, com outro Link State ID.


### Questão 5: o DR numa rede com quatro routers (192.168.100.0/24)

**O que se pede.** Em R3, R4, R5 e R6 correr `show ip ospf interface` na interface ligada à 192.168.100.0/24. Dizer quantas adjacências cada um tem e relacionar com o seu papel (5a); explicar para que serve o DR (5b).

> **Conceito: adjacência não é o mesmo que vizinho.** Dois routers são **vizinhos** (*neighbor count*) quando se veem por Hello. Só se tornam **adjacentes** (*adjacent neighbor count*) quando sincronizam as bases de dados e chegam a Full. Numa rede com DR, só o DR e o BDR são adjacentes com todos. Dois routers que não são DR nem BDR (**DROTHER**) são vizinhos mas ficam em **2-Way**, sem adjacência.

#### outRes

![R3](evidencias/t2_q5_r3.png)

*R3 (Fa2/0, 192.168.100.1): State DR, 3 vizinhos, 3 adjacências.*

![R4](evidencias/t2_q5_r4.png)

*R4 (Fa1/0, 192.168.100.253): State DROTHER, 3 vizinhos, 2 adjacências.*

![R5](evidencias/t2_q5_r5.png)

*R5 (Fa1/0, 192.168.100.254): State BDR, 3 vizinhos, 3 adjacências.*

![R6](evidencias/t2_q5_r6.png)

*R6 (Fa1/0, 192.168.100.2): State DROTHER, 3 vizinhos, 2 adjacências.*

#### 5a) Adjacências por router (texRes)

| Router | Router ID | Papel | Neighbor Count | Adjacent neighbor count | Estado com cada vizinho |
|---|---|---|---|---|---|
| R3 | 2.0.0.3 | **DR** | 3 | **3** | Full com R4, R5 e R6 |
| R5 | 2.0.0.5 | **BDR** | 3 | **3** | Full com R3, R4 e R6 |
| R4 | 2.0.0.4 | DROTHER | 3 | **2** | Full com R3 (DR) e R5 (BDR); **2-Way** com R6 |
| R6 | 2.0.0.6 | DROTHER | 3 | **2** | Full com R3 (DR) e R5 (BDR); **2-Way** com R4 |

Os `show ip ospf neighbor` confirmam: no R4 aparece `2WAY/DROTHER` para o R6, e no R6 aparece `2WAY/DROTHER` para o R4. As linhas extra do R3 (R2 na Fa1/1) e do R6 (R7 na Fa1/1 e R8 na Se2/0) são de **outras redes**: o *neighbor count* de cada interface só conta a sua rede.

O DR é o R3 (ID 2.0.0.3) e não o R6 (ID 2.0.0.6, o maior). Isto confirma que a eleição **não é preemptiva**: ficam DR e BDR os routers que estavam ativos primeiro, e o R6, que chegou depois, não os substituiu apesar do ID maior.

> **5a (texRes).** O **DR (R3)** e o **BDR (R5)** têm **3 adjacências** cada, uma com cada outro router da rede (Full com todos). Os dois **DROTHER (R4 e R6)** têm só **2 adjacências**: Full com o DR e com o BDR, e **2-Way** (apenas vizinhos, sem sincronizar bases de dados) entre si. O número de adjacências depende, portanto, do papel: DR e BDR são adjacentes de todos, os restantes só do DR e do BDR. Com 4 routers isto dá 5 adjacências completas na rede em vez de 6 (n(n-1)/2 de uma malha completa). A poupança cresce depressa, já que 10 routers dariam 45 adjacências numa malha completa contra cerca de 17 com DR e BDR.

#### 5b) Para que serve o DR (texRes)

> **5b (texRes).** O DR existe para **reduzir o custo do OSPF numa rede de acesso múltiplo**. (1) **Menos adjacências**: cada router só sincroniza as bases de dados com o DR e o BDR, em vez de com todos, o que reduz o tráfego de sincronização e o processamento. (2) **Flooding centralizado**: um router que muda um LSA envia-o ao DR e ao BDR (224.0.0.6, AllDRouters) e o DR retransmite-o a toda a rede (224.0.0.5, AllSPFRouters), evitando que todos inundem a todos. (3) **Representação única da rede**: o DR origina o Network-LSA, que diz quais são os routers ligados ao segmento. O **BDR** acompanha tudo em segundo plano e assume o papel imediatamente se o DR falhar, sem esperar pela reconstrução das adjacências.


### Questão 6: cortar a ligação R6-R7 e a impossibilidade do desvio

**O que se pede.** Fazer um traceroute do Terminal 1 para o Terminal 2; cortar a ligação R6-R7 para que o tráfego passe por R8 e R9 e repetir o traceroute (2 × outRes). No R6 correr `show ip ospf database` (6a) e, com base nesse resultado, justificar a impossibilidade de fazer a alteração pedida (6b). No fim, repor a ligação.

> **Conceito: regras que o OSPF impõe entre áreas.**
>
> 1. **Todo o tráfego entre áreas passa pela área 0 (backbone).** As áreas não se ligam diretamente umas às outras.
> 2. Só um **ABR**, isto é, um router **ligado à área 0** e a outra área, resume a informação de uma área para outra, com Summary-LSA (tipo 3).
> 3. Um ABR ligado ao backbone só considera, no cálculo de rotas inter-área, os Summary-LSA que vêm da **área 0** (RFC 2328, secção 16.2). Isto evita ciclos entre áreas.

#### outRes: traceroutes

![Traceroute antes do corte](evidencias/t2_q6_trace_antes.png)

*Antes do corte: R1, R2, R3, R6 (192.168.100.2), R7 (172.20.1.2) e Terminal 2.*

![Traceroute com a ligação cortada](evidencias/t2_q6_trace_corte.png)

*Com a ligação cortada: o salto 2 é o próprio R1 (192.168.1.1), a responder `Destination host unreachable`.*

O R1 não tem rota para 172.16.1.0/24 (confirmado com `show ip route` no R1). Isto quer dizer que a rota **desapareceu ao longo da cadeia** que a trazia até ele: R7 e R6 (área 1), R6 e R3 (área 0), R2 (redistribuição para o AS1) e R1.

#### 6a) `show ip ospf database` no R6 (outRes)

![Base de dados do R6, área 1](evidencias/t2_q6_r6_database_area1.png)

*Área 1 na base de dados do R6 depois do corte.*

O que se vê na área 1:

| Secção | Conteúdo | Leitura |
|---|---|---|
| Router Link States | **R6 (2.0.0.6): Link count 0**; R7: 3 ligações; R9: 1 ligação | O R6 já não tem nenhuma interface ativa na área 1. Os LSA do R7 e do R9 ficam na tabela por resíduo, mas o R6 já não os alcança |
| Net Link States | 172.20.1.2 (R7) e 172.20.1.6 (R9) | Network-LSA dos dois segmentos da área 1. O DR da 172.20.1.0/30 passou a ser o R7 quando o R6 saiu |
| Summary Net Link States | 172.17.0.0, 172.18.1.0, 172.20.1.8, 172.20.1.12, 192.168.50.0, 192.168.60.0 e 192.168.100.0, **todos do R6** | O R6 anuncia à área 1 as redes de fora dela |
| Summary ASB Link States | 2.0.0.2, do R6 | O R6 anuncia à área 1 como chegar ao ASBR R2 |

A rede 172.16.1.0/24 não aparece nos Summary porque é **interna à área 1**: está só no Router-LSA do R7 (como ligação *Stub*).

> *Nota.* O print mostra apenas a área 1. As secções das áreas 0 e 2 não foram capturadas e a justificação seguinte apoia-se nas regras do OSPF.

#### 6b) Justificação da impossibilidade (texRes)

Fisicamente o caminho alternativo existe: R6, ligação série ao R8, R9 e a Fa1/0 do R7. Mas o OSPF não o consegue usar:

1. A rede 172.16.1.0/24 pertence à **área 1**. Para o R6 chegar lá por R8 e R9, a informação teria de passar da área 1 para a área 2 e daí para o R6. O único router que o poderia fazer é o **R9**, que tem interfaces nas áreas 1 e 2.
2. O **R9 não está ligado à área 0**. Um router só se comporta como ABR (origina Summary-LSA entre áreas) se estiver ligado ao backbone. O R9 é só um router interno das duas áreas e **não resume** uma para a outra.
3. Mesmo que um Summary-LSA chegasse ao R6 pela área 2, o R6, ligado ao backbone, **ignora** os Summary-LSA que não vêm da área 0.

Efeito em cadeia: o R6 perde a rota, deixa de anunciar à área 0 as redes da área 1 que dependiam do R7, o R3 e o R2 perdem-nas, o R2 deixa de as redistribuir (tipo 5) para o AS1 e o R1 fica sem rota.

> **6b (texRes).** Com a ligação R6-R7 cortada, o Terminal 1 deixa de alcançar o Terminal 2: o R1 responde "Destination host unreachable" por já não ter rota para 172.16.1.0/24. Existe um caminho físico alternativo (R6, R8, R9, R7), mas o OSPF não o consegue usar. A base de dados do R6 mostra que, na área 1, o Router-LSA do próprio R6 ficou sem ligações (`Link count 0`) e que os LSA do R7 e do R9 já não são alcançáveis a partir dele. A rede do Terminal 2 pertence à área 1, e só um ABR a poderia anunciar à área 2. O único candidato é o R9, mas **não está ligado à área 0**, pelo que não atua como ABR e não origina Summary-LSA entre as áreas 1 e 2. Além disso, o R6, ligado ao backbone, só considera Summary-LSA da área 0 no cálculo de rotas inter-área (RFC 2328, secção 16.2). O R6 perde assim a rota para 172.16.1.0/24 e retira os resumos que anunciava à área 0, o R3 e o R2 perdem-nos, o R2 deixa de os redistribuir para o AS1 e o R1 fica sem rota. A alteração pedida é impossível com esta topologia, porque a área 1 só tem uma ligação ao backbone (através do R6) e o percurso por R8 e R9 liga duas áreas entre as quais nenhum ABR faz a ponte. Só seria possível com uma ligação adicional à área 0 (por exemplo um *virtual link* entre R6 e R9 através da área 2).

#### Reposição da ligação

![Traceroute depois de repor a ligação](evidencias/t2_q6_trace_reposto.png)

*Ligação reposta: o traceroute volta a fazer os 6 saltos pelo percurso original.*

> **Nota prática (lição aprendida).** O corte foi feito **suspendendo a ligação no GNS3**. Isto **não** baixa a interface no router, que continua `up/up` e só perde o vizinho ao fim do Dead timer de 40 s. Depois de retomar a ligação, a adjacência R6-R7 não recuperou de imediato (o R6 continuava sem ligações na área 1 e o traceroute continuava a falhar) e só voltou ao normal depois de se forçar a reposição da interface do R6 (por exemplo `shutdown` seguido de `no shutdown` na Fa1/1). Para esta questão é preferível cortar com `shutdown` no R6 (como no roteiro da secção 6), porque o router apanha a falha de imediato e repõe a adjacência sem surpresas.


### Questão 7: caminhos diferentes na ida e na volta do ping

**O que se pede.** Iniciar capturas nas interfaces 172.20.1.2 e 172.20.1.5 do R7. Fazer ping do Terminal 2 para 172.17.0.1 (a LAN do R9) e dizer qual o caminho do echo request e do echo reply e porquê (7a). Tentar alterar o custo da interface 172.20.1.6 do R9 para que o reply faça o caminho inverso do request (7b).

> **Conceito: o OSPF escolhe o tipo de rota antes da métrica.** Para o mesmo destino, o OSPF prefere, por esta ordem: **intra-área** (O), **inter-área** (O IA), **externa tipo 1** (E1) e **externa tipo 2** (E2). A métrica só desempata entre rotas **do mesmo tipo**. Por isso uma rota intra-área ganha a uma inter-área mesmo que seja muito mais cara.

#### 7a) O caminho do ping

Observado nas capturas do R7:

| Pacote | Sai do R7 por | Entra no R7 por |
|---|---|---|
| **Echo request** (172.16.1.2 para 172.17.0.1) | **172.20.1.2** (Fa1/0, para o R6) | |
| **Echo reply** (172.17.0.1 para 172.16.1.2) | | **172.20.1.5** (Fa1/1, vinda do R9) |

Percurso: **ida** Terminal 2, R7, R6, R8, R9. **Volta** R9, R7, Terminal 2. O percurso é **assimétrico**: a volta é direta e a ida dá a volta por R6 e R8.

**capRes (a anexar):** `evidencias/t2_q7_r7_172.20.1.2.png` (echo request) e `evidencias/t2_q7_r7_172.20.1.5.png` (echo reply), com o filtro `icmp` no Wireshark.

**Porquê (cada extremo aprende a rede do outro de forma diferente):**

- **Ida, no R7.** A rede 172.17.0.0/16 está na **área 2**. O R7 só a conhece por um **Summary-LSA do R6** (o ABR), como mostra a base de dados da questão 6 (`172.17.0.0` anunciada por 2.0.0.6 na área 1). O R9, apesar de ligado às áreas 1 e 2, não é ABR (não tem ligação à área 0) e não resume a área 2 para a área 1. Logo, a única rota do R7 para a LAN do R9 é **inter-área, via R6**, e atravessa a série até ao R8 e depois R9. Custo no R7: 10 (R7 até R6) + 647 (série) + 10 (R8 até R9) + 10 (LAN do R9) = **677**.
- **Volta, no R9.** A rede 172.16.1.0/24 está na **área 1**, a mesma a que o R9 está ligado pela Fa1/0. O R9 conhece-a como rota **intra-área**, diretamente pelo R7: custo 10 (Fa1/0) + 10 (LAN do R7) = **20**.

> **7a (texRes).** O **echo request** segue Terminal 2, R7, R6, R8, R9: sai do R7 pela interface 172.20.1.2 em direção ao R6, atravessa a ligação série até ao R8 e chega ao R9 (172.17.0.1). O **echo reply** regressa pelo caminho mais curto, diretamente do R9 ao R7, e entra no R7 pela interface 172.20.1.5. A razão é a forma como cada router aprende a rede do outro. O R7 só conhece 172.17.0.0/16 através de um Summary-LSA originado pelo R6, que é o ABR entre a área 1 e a área 2: o R9, embora esteja nas duas áreas, não tem ligação à área 0, não é ABR e não resume uma área para a outra. A rota do R7 é assim inter-área e passa por R6, R8 e R9 (custo 677). Já o R9 conhece 172.16.1.0/24 como rota intra-área, pela área 1, diretamente através do R7 (custo 20). Resulta um percurso assimétrico, em que ida e volta usam ligações diferentes.

#### 7b) Alterar o custo na interface 172.20.1.6 do R9

Para o reply fazer o caminho inverso (R9, R8, R6, R7), o R9 teria de deixar de preferir a Fa1/0. Aumentou-se o custo da Fa1/0 do R9 (de 10 para **500**) para a tornar **menos atrativa**, e o echo reply continuou a regressar pelo R7 diretamente.

Os custos em jogo, vistos do R9 para 172.16.1.0/24:

| Caminho | Tipo de rota | Custo |
|---|---|---|
| Direto pelo R7 (Fa1/0) | **Intra-área** | Fa1/0 + 10 (LAN do R7) = 20 com custo 10, 510 com custo 500 |
| Via R8, R6 e R7 | **Inter-área** | 10 (R9 até R8) + 647 (série) + 10 (R6 até R7) + 10 (LAN do R7) = **677** |

Repare que 500 não ultrapassa a via alternativa (510 < 677), por isso este teste, sozinho, não distingue "falta de custo" de "regra do tipo de rota". Para a prova ser limpa, o custo da Fa1/0 teria de passar de 667 (por exemplo 1000): a rota direta custaria 1010, mais cara do que 677, e mesmo assim o R9 manteria a intra-área.

> **7b (texRes).** Não foi possível inverter o percurso do echo reply alterando o custo da interface 172.20.1.6 do R9. Aumentámos o custo da Fa1/0 para a tornar menos atrativa e o echo reply continuou a regressar diretamente pelo R7. O R9 conhece 172.16.1.0/24 como rota **intra-área** (área 1), e conhece, ou poderia conhecer, uma alternativa **inter-área** via R8 e R6, anunciada por Summary-LSA do R6 (custo 677). O OSPF escolhe primeiro pelo **tipo de rota** (intra-área, inter-área, externa tipo 1, externa tipo 2) e só depois pela métrica, pelo que uma rota intra-área é sempre preferida a uma inter-área, qualquer que seja o seu custo, mesmo que seja maior. Mudar o custo só altera o valor da métrica da rota escolhida, nunca o seu tipo. Só deixando de existir a rota intra-área (por exemplo desativando a interface Fa1/0 do R9) é que o R9 passaria a usar o caminho alternativo, e isso já não é uma alteração de custo.

> **Reposição.** Depois do teste, repor o custo com `no ip ospf cost` na Fa1/0 do R9 e confirmar com `show ip route 172.16.1.0` que a métrica voltou a 20.


### Resumo: as ideias que as sete questões ensinam

| Questão | Ideia-chave |
|---|---|
| 1 | Sincronizar bases de dados em três passos: **DBD** (índice), **LSR** (pedido do que falta) e **LSU** (a resposta). Só se chega a Full quando não falta nada |
| 2 | O ABR origina **um Router-LSA por área**. Ethernet dá ligação **Transit**, série dá **point-to-point** mais **Stub** |
| 3 | A falha é detetada pelo **Dead timer** (silêncio). O Hello com o nosso ID na lista de vizinhos prova a comunicação **bidirecional**. Cada tipo de LSA tem o seu âmbito: 1 e 2 ficam na área, 3 e 4 atravessam áreas pelo ABR, 5 circula no AS |
| 4 | O DR só existe em redes de **difusão**. O **Network-LSA** é originado pelo DR, e só quando tem vizinhos Full |
| 5 | DR e BDR são adjacentes de todos, os DROTHER só do DR e do BDR (2-Way entre si). A eleição **não é preemptiva** |
| 6 | O tráfego entre áreas passa pela **área 0**. Sem ligação ao backbone, o R9 não é ABR e o desvio por R8 e R9 não existe |
| 7 | O OSPF prefere **intra-área a inter-área** antes de olhar à métrica. Por isso o custo não inverte o percurso da volta |


## 8. Dúvidas para a aula

Perguntas surgidas durante o trabalho, para esclarecer com o professor. A resposta provisória é a que se obteve durante a análise e fica **por confirmar**.

### Dúvida 1: diferença entre ligação ponto a ponto e ligação Transit (Router-LSA)

**Contexto (questão 2).** No Router-LSA do R6, a área 1 (Fa1/1, Ethernet) mostra uma única ligação *Transit Network* com Link ID 172.20.1.1 e Link Data 172.20.1.1. A área 2 (Se2/0, série) mostra uma ligação *point-to-point* com Link ID 2.0.0.8 e Link Data 172.20.1.14, mais uma ligação *stub* para 172.20.1.12/30. Na área 1 não aparece o router ID do vizinho (R7).

**Pergunta.** Qual é a diferença entre uma ligação ponto a ponto e uma ligação Transit, e porque é que o vizinho só aparece nomeado na primeira?

**Resposta provisória.**

| | Ponto a ponto | Transit |
|---|---|---|
| Rede típica | Série, ou Ethernet configurada como ponto a ponto | Ethernet (difusão) ou NBMA |
| Routers na ligação | 2 | 2 ou mais |
| DR/BDR | Não há | Há |
| Network LSA | Não existe | Existe, originado pelo DR |
| Link ID | Router ID do vizinho | Endereço da interface do DR |
| Link Data | IP da interface local | IP da interface local |
| Sub-rede | Ligação *stub* separada no Router-LSA | Descrita no Network LSA (máscara e routers ligados) |

Na ligação ponto a ponto o grafo tem uma aresta entre dois routers. Na Transit a rede é um nó extra do grafo, representado pelo DR, e cada router liga-se a esse nó. Por isso o R7 só aparece no Network LSA do DR (`show ip ospf database network 172.20.1.1`), e não no Router-LSA do R6.

**Para confirmar com o professor.**

- Se o R6 é DR da rede 172.20.1.0/30, o Link ID e o Link Data da ligação Transit coincidem (ambos 172.20.1.1). Esta leitura está correta?
- Faz sentido configurar a Fa1/1 como `ip ospf network point-to-point` nas ligações entre apenas dois routers? Que efeito teria nos LSA e na eleição do DR?

**Atualização (depois das questões 4 a 6).**

- Na questão 5, o `show ip ospf neighbor` do R6 mostrou o R7 como `FULL/BDR` na Fa1/1, o que confirma que o **R6 era o DR** da rede 172.20.1.0/30 quando o Router-LSA da questão 2 foi capturado (daí Link ID = Link Data = 172.20.1.1). A primeira pergunta fica **respondida**.
- Depois do corte e reposição da questão 6, o Network-LSA dessa rede passou a ser originado pelo R7 (Link State ID 172.20.1.2). O DR depende da ordem de arranque, porque a eleição não é preemptiva.
- A segunda pergunta (efeito de `ip ospf network point-to-point` nas ligações entre apenas dois routers) continua por confirmar com o professor.
